In [ ]:
import geopandas as gpd
import osmnx as ox


#Load data
stops = gpd.read_file(r"C:\Users\luca\Downloads\Lux_public_transport_project\stop_freq_avl.gpkg")

#Similar to the network formation, we are only concerned with P+R stations within the AVL network, buffer distance of 600m
stops_2169 = stops.to_crs(epsg=2169)

stop_area = stops_2169.buffer(400).union_all()

stop_area_4326 = gpd.GeoSeries([stop_area], crs="EPSG:2169").to_crs(epsg=4326).iloc[0]

PR_filters = {"park_ride": "yes"}

PR_stations = ox.features_from_polygon(
    stop_area_4326,
    tags=PR_filters,
)
PR_stations = PR_stations[PR_stations.index.get_level_values("element") == "way"]
PR_stations = PR_stations[PR_stations["name"].notna()].drop_duplicates()

#Create a 250m buffer and find all stops within that buffer
PR_stations_2169 = PR_stations.to_crs(epsg=2169)
PR_stations_2169["geometry"] = PR_stations_2169.geometry.centroid.buffer(250)

PR_stops = gpd.sjoin(
    stops_2169,
    PR_stations_2169,
    predicate="within"
)

#Assign standardised names to P+R stations
PR_stops["name"] = PR_stops["name"].replace({
    "P+R Luxembourg Sud - Parking A": "Howald",
    "P+R Luxembourg Sud - Parking B": "Howald",
    "Bouillon": "Hollerich",
    "P&R Kockelscheuer": "Kockelscheuer",
    "P+R Héienhaff": "Héienhaff",
    "P+R Stade de Luxembourg": "Stadion"
})
PR_stops = PR_stops.drop_duplicates(subset=["stop_id", "name"])

#Save only the stop_ID, name and geometry — reproject to 4326 for QGIS
PR_stops = PR_stops[["stop_id", "name", "stop_lat", "stop_lon", "geometry"]].reset_index(drop=True).to_crs(epsg=4326)
PR_stops["node"] = "pt_" + PR_stops["stop_id"].astype(str)


PR_stops.to_file(
    r"C:\Users\luca\Downloads\Lux_public_transport_project\PR_stops.gpkg",
    driver="GPKG"
)
print('P+R stops saved')

P+R stops saved
